# FADE Quickstart

**Frequency-Adaptive Decay Encoding** — drop-in KV cache compression for HuggingFace transformers.

This notebook demonstrates the three presets (`safe`, `balanced`, `aggressive`) on Qwen2.5-0.5B-Instruct and compares KV cache size vs a plain FP16 baseline.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omodaka9375/fade/blob/main/examples/quickstart.ipynb)

In [ ]:
# Install FADE (run once)
!pip install -q git+https://github.com/Omodaka9375/fade.git

In [ ]:
import torch
from transformers import DynamicCache
from fade import FadeConfig, create_tiered_cache
from fade.eval.memory import cache_storage_bytes
from fade.patch import load_model

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
PROMPT = "Explain how a CPU cache hierarchy works, in three paragraphs."
MAX_NEW = 128

print(f"Device: {DEVICE}")
model, tokenizer = load_model(MODEL_ID, device_map=DEVICE, dtype=DTYPE, attn_impl="sdpa")
print(f"Model loaded: {MODEL_ID}")

In [ ]:
def run(label, config=None):
    enc = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
    cache = DynamicCache() if config is None else create_tiered_cache(model, dtype=DTYPE, config=config)
    out = model.generate(**enc, past_key_values=cache, max_new_tokens=MAX_NEW, do_sample=False)
    text = tokenizer.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True)
    kv_mib = cache_storage_bytes(cache) / (1024 * 1024)
    print(f"\n{'='*60}")
    print(f"  {label}  |  kv_cache = {kv_mib:.2f} MiB")
    print(f"{'='*60}")
    print(text[:300])
    return kv_mib

In [ ]:
base = run("Baseline (FP16)")

In [ ]:
safe = run("FadeConfig.safe()     ~3-4x", FadeConfig.safe())

In [ ]:
bal = run("FadeConfig.balanced()  ~5x", FadeConfig.balanced())

In [ ]:
agg = run("FadeConfig.aggressive() ~7-8x", FadeConfig.aggressive())

In [ ]:
print(f"\n{'='*60}")
print(f"  Summary")
print(f"{'='*60}")
print(f"Baseline:   {base:.2f} MiB")
print(f"Safe:       {safe:.2f} MiB  ({100*(1-safe/base):.0f}% smaller)")
print(f"Balanced:   {bal:.2f} MiB  ({100*(1-bal/base):.0f}% smaller)")
print(f"Aggressive: {agg:.2f} MiB  ({100*(1-agg/base):.0f}% smaller)")